# Lesson 23 Lab — Accuracy Regression Tests for Quantized Models

**Puzzle:** Can one aggregate score hide a serious quantization regression?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Quantization quality is not one cosine score. A release can preserve average logits while changing top-1 decisions, rare domains, long-context behavior, calibration-sensitive layers, or safety-critical outputs. Regression testing turns those failure modes into frozen gates that can block a numerically small but behaviorally important change.


## 0. Predict before running

1. Predict whether cross-entropy, perplexity, and top-1 agreement will all move in the same relative direction.
2. Explain why the synthetic perplexity magnitude is not meaningful as a language-model score.
3. Design at least three deployment slices that an aggregate metric could hide.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Quality evidence spans token likelihood (cross-entropy/perplexity), task metrics, output/logit agreement, safety/alignment cases, and business-specific slices.

- Perplexity measures token likelihood, task accuracy measures decisions, and alignment samples cover product behavior.
- Thresholds should be frozen before examining the candidate.
- Slice-level failures can be hidden by a stable global average.


## 2. Derive the mechanism

Perplexity is `exp(mean token cross-entropy)`; a small average loss change can coexist with large ranking changes on a rare slice. Top-1 agreement reveals decision changes but not whether either answer is correct.

For targets y and logits z, cross-entropy measures probability assigned to y; perplexity is `exp(loss)` and can magnify small loss changes. Top-1 agreement instead asks whether the candidate preserves the baseline decision, regardless of whether either decision is correct. Logit distance, task accuracy, exact-match, calibration, and human/safety checks answer still different questions.

A release gate should define baselines, datasets, seeds, tolerances, and slice policies before the candidate is evaluated. Otherwise thresholds drift to accommodate the observed regression.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "23-accuracy-regression"
device = require_cuda()
torch.manual_seed(2026 + 23)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | floating-point synthetic classifier logits over 4,096 tokens |
| Candidate | INT4-dequantized weight logits for the same hidden states and targets |
| Held constant | tokens, vocabulary, targets, hidden states, weight matrix, seed |
| Measurements | loss, derived perplexity, overall and half-slice top-1 agreement |
| Evidence | `pytorch-gpu` |

**Experiment:** Run a tiny CUDA language-model head before and after INT4 weight Q/DQ, then compare cross-entropy, perplexity, top-1 agreement, and slice metrics.


## 5. Read the experiment code

The CUDA probe computes loss, perplexity, overall agreement, and two slices from identical hidden states before and after INT4 Q/DQ.

The notebook generates one fixed synthetic classification problem, computes baseline and quantized logits, and evaluates the same targets. It reports the complete set and two halves so a slice disagreement cannot be hidden by one aggregate.

Because random logits yield enormous losses and perplexities, the absolute values are intentionally labeled synthetic. The exercise demonstrates metric relationships and gate structure, not language modeling ability.

Only after these variables match the protocol should the cell be executed.


In [2]:
torch.manual_seed(7); vocab,hidden,tokens=256,128,4096; h=torch.randn(tokens,hidden,device=device); w=torch.randn(vocab,hidden,device=device); targets=torch.randint(0,vocab,(tokens,),device=device)
base=h@w.t(); _,_,dq=symmetric_quantize(w,bits=4,group_size=64); cand=h@dq.t()
base_loss=torch.nn.functional.cross_entropy(base,targets); cand_loss=torch.nn.functional.cross_entropy(cand,targets)
slices={"first_half":slice(0,tokens//2),"second_half":slice(tokens//2,None)}; slice_rows={}
for name,s in slices.items(): slice_rows[name]={"top1_agreement":round((base[s].argmax(-1)==cand[s].argmax(-1)).float().mean().item(),6)}
result=base_result(23,"pytorch-gpu"); result.update({"synthetic_probe":{"tokens":tokens,"vocab":vocab,"baseline_loss":round(base_loss.item(),7),
    "candidate_loss":round(cand_loss.item(),7),"baseline_perplexity":round(base_loss.exp().item(),5),"candidate_perplexity":round(cand_loss.exp().item(),5),
    "top1_agreement":round((base.argmax(-1)==cand.argmax(-1)).float().mean().item(),6),"slices":slice_rows},
    "conclusion":"Multiple frozen metrics exposed the synthetic INT4 regression; they are not scores for a named language model."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Baseline loss | 32.049492 |
| Candidate loss | 32.212620 |
| Baseline synthetic perplexity | 8.297e+13 |
| Candidate synthetic perplexity | 9.767e+13 |
| Top-1 agreement | 83.6914% |


## 7. Interpret rather than merely print

Candidate loss increased from 32.049492 to 32.212620. Exponentiation turned that modest difference into synthetic perplexities of about `8.30e13` and `9.77e13`. Overall top-1 agreement was 0.836914; the two halves were 0.838379 and 0.835449.

The near-equal slices do not reveal a concentrated failure in this constructed set, but roughly 16% decision disagreement is clearly visible. A real release would need task correctness, not only agreement with the baseline.

**Inspection rule:** Apply predeclared gates to every metric and slice. This synthetic probe is not a benchmark score for a named LLM.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Multiple frozen metrics exposed the synthetic INT4 regression; they are not scores for a named language model.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:09+00:00",
  "lesson": 23,
  "schema_version": 1,
  "synthetic_probe": {
    "baseline_loss": 32.0494919,
    "baseline_perplexity": 82969305808896.0,
    "candidate_loss": 32.2126198,
    "candidate_perplexity": 97670416826368.0,
    "slices": {
      "first_half": {
        "top1_agreement": 0.838379
      },
      "second_half": {
        "top1_agreement": 0.835449
      }
    },
    "tokens": 4096,
    "top1_agreement": 0.836914,
    "vocab": 256
  }
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Use a layered quality gate and retain the baseline outputs needed to explain a regression.

**Acceptance/rollback:** Freeze datasets, prompts, decoding, baseline revision, thresholds, and slice definitions. Fail on a critical slice even if the global average passes.

**Failure analysis:** Perplexity can overflow or become hard to interpret at extreme synthetic losses. Baseline agreement can preserve a baseline mistake, and average accuracy can hide a critical slice. Reusing calibration prompts for regression also lets quantizer selection overfit the gate.


## 10. Extend the evidence

Replace synthetic logits with a small named model and a frozen, redistribution-safe suite: perplexity on held-out text, task accuracy, long-context slices, multilingual/code/tool-use samples, and answer/logit agreement. Publish thresholds and reversal criteria before running the candidate.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
